# TaFi Video Studio — Colab GPU Edition
Chạy toàn bộ pipeline trên **GPU miễn phí (Colab T4)**:
- ASR: **SenseVoice** (mặc định FunASR GPU; có thể đổi sang sherpa CPU nhanh)
- OCR: **PP-OCRv5 server** + CUDA
- Render: **h264_nvenc** (GPU) — video 2 tiếng chỉ ~5–10 phút
- Giữ nguyên: dịch xKiro + QC chính tả + CapCut TTS + dictionary gate

Cách dùng: Runtime ▸ Run all. Chạy xong copy link `https://xxx.trycloudflare.com` mở web.
Chú ý: phiên Colab tự mất sau ~vài giờ/đóng tab → nên gắn Drive và xuất file về máy trước khi tắt.


In [ ]:
# 1) Kiểm tra GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv 2>/dev/null || echo "Không có GPU — vẫn chạy được nhưng chậm hơn"
import torch
print('torch CUDA:', torch.cuda.is_available())

In [ ]:
# 2) Gắn Google Drive (lưu video/audio/phụ đề) — tùy chọn
from google.colab import drive
MOUNT_DRIVE = True  # @param {type:"boolean"}
if MOUNT_DRIVE:
    drive.mount('/content/drive')
    print('Đã gắn: /content/drive')

In [ ]:
# 3) Lấy code (mặc định: tự clone repo GitHub chứa sẵn code + zip; để trống GIT_URL nếu muốn upload zip thủ công)
import os, zipfile, glob
GIT_URL = "https://github.com/theandanh000-cmyk/tafi-colab-gpu.git"  # @param {type:"string"} — mặc định: tự clone repo (kèm zip); để trống nếu muốn upload zip thủ công
if GIT_URL.strip():
    os.system(f"cd /content && git clone {GIT_URL.strip()} tafi 2>/dev/null || true")
else:
    from google.colab import files
    print(">> Bấm chọn file TaFi-VS-Tool-Web.zip trong hộp thoại")
    up = files.upload()
    fn = list(up.keys())[0]
    os.makedirs('/content/tafi_root', exist_ok=True)
    with zipfile.ZipFile(fn) as z:
        z.extractall('/content/tafi_root')
    inner = glob.glob('/content/tafi_root/*/server.js')
    src = os.path.dirname(inner[0]) if inner else '/content/tafi_root'
    os.system(f"rm -rf /content/tafi && cp -r '{src}' /content/tafi")
os.chdir('/content/tafi')
print('Code tại:', os.getcwd())
print('server.js:', os.path.exists('server.js'))

In [ ]:
# 4) Cài môi trường
!apt-get -qq update >/dev/null 2>&1 && apt-get -qq install -y ffmpeg >/dev/null 2>&1
!pip -q install numpy pillow opencv-python-headless rapidocr opencc-python-reimplemented spylls you-get curl_cffi yt-dlp sherpa-onnx
# GPU cho OCR (thay onnxruntime bằng bản GPU)
!pip -q install --upgrade onnxruntime-gpu
# ASR GPU bằng FunASR SenseVoice (Colab đã có sẵn torch CUDA)
!pip -q install funasr modelscope
!cd /content/tafi && (cd pipeline/gemini_translator && npm install --omit=dev 2>&1 | tail -1)
print('Đã cài xong deps')

In [ ]:
# 5) Tải model ASR SenseVoice (sherpa int8, dùng khi đổi ASR_ENGINE=sensevoice)
!mkdir -p /content/tafi/pipeline/models
!cd /content/tafi/pipeline/models && { test -f sherpa-onnx-sense-voice-zh-en-ja-ko-yue-int8-2024-07-17/model.int8.onnx || { curl -sL -o sv.tar.bz2 https://github.com/k2-fsa/sherpa-onnx/releases/download/asr-models/sherpa-onnx-sense-voice-zh-en-ja-ko-yue-int8-2024-07-17.tar.bz2 && tar xjf sv.tar.bz2 && rm -f sv.tar.bz2; }; }
print('Model sherpa sensevoice sẵn sàng')

In [ ]:
# 6) Khởi động server với cấu hình GPU
import os, subprocess, time
os.environ.update({
    'ASR_ENGINE': 'sensevoice-funasr',   # GPU thật (FunASR); đổi 'sensevoice' = sherpa CPU (nhanh, đỡ tài nguyên)
    'OCR_MODEL': 'ppocrv5-server',       # OCR chuẩn nhất, chạy CUDA
    'OCR_USE_CUDA': '1',
    'RENDER_CODEC': 'h264_nvenc',        # render GPU
    'PORT': '3000',
    # Engine dịch mặc định: xKiro (free, không cần GPU) — GPU chỉ dùng cho ASR/OCR/render.
    'TRANSLATE_ENGINE': 'xkiro',
})
os.system("cd /content/tafi && pkill -f 'node server.js' 2>/dev/null; nohup node server.js > /tmp/tafi.log 2>&1 &")
time.sleep(6)
print(os.popen("curl -s localhost:3000/api/health | head -c 120").read())

In [ ]:
# 7) Mở web ra internet (bắt buộc có tunnel trên Colab)
!which cloudflared >/dev/null 2>&1 || (curl -sL https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -o /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared)
import subprocess, re
proc = subprocess.Popen(['cloudflared','tunnel','--url','http://localhost:3000','--no-autoupdate'],
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
url = None
for line in proc.stdout:
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', line)
    if m:
        url = m.group(0)
        break
print('🌐 LINK WEB (mở trên điện thoại/máy):', url)
open('/tmp/tunnel_url.txt','w').write(url or '')

In [ ]:
# 8) Giữ phiên sống (tự bấm giữ mỗi 60 phút) — để chạy lâu không bị Colab ngắt
import time, subprocess
for i in range(600):
    time.sleep(60)
    subprocess.run(['curl','-s','-o','/dev/null','localhost:3000/api/health'])
    print(f'giữ phiên {i+1}/600 phút — link web vẫn ở /tmp/tunnel_url.txt', flush=True)

## Ghi chú
- ASR mặc định `sensevoice-funasr` = FunASR + SenseVoiceSmall chạy CUDA. Nếu cài funasr lỗi, đổi `ASR_ENGINE=sensevoice` (sherpa-onnx CPU, vẫn nhanh ~25–50x real-time).
- OCR: `ppocrv5-server` tự tải lần đầu (~165MB) vào thư mục rapidocr; chỉ quét vùng đáy + bỏ khung lặp như bản local.
- Render: `h264_nvenc` tự dùng GPU; không có GPU thì đặt `RENDER_CODEC=libx264`.
- Key xKiro: đặt file `pipeline/gemini_translator/xkiro_key.txt` (hoặc env `XKIRO_API_KEY`) trước khi dịch.
- File output nằm trong `/content/tafi/jobs/<jobId>/` — chạy cell Drive để copy về Drive trước khi hết phiên.